In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/Antlings_Visdrone/archive.zip -d /content/drive/MyDrive/Antlings_Visdrone/visdrone

replace /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-test-challenge/images/0000000_00098_d_0000001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
from pathlib import Path

RAW_ROOT = Path('/content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset')
CUSTOM_ROOT = Path('/content/drive/MyDrive/Antlings_Visdrone/visdrone_custom')
YAML_PATH = Path('/content/drive/MyDrive/Antlings_Visdrone/data.yaml')

print(RAW_ROOT)
print(CUSTOM_ROOT)
print(YAML_PATH)

/content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset
/content/drive/MyDrive/Antlings_Visdrone/visdrone_custom
/content/drive/MyDrive/Antlings_Visdrone/data.yaml


In [ ]:
import os

print(f"--- Inspecting Raw VisDrone Dataset Structure ---\n")

# Inspect 'train' split parent directory
train_split_dir = RAW_ROOT / 'VisDrone2019-DET-train'
print(f"Checking contents of training split directory: {train_split_dir}")
if train_split_dir.exists():
    print(f"  Contents: {os.listdir(train_split_dir)}")
else:
    print(f"  Directory does not exist: {train_split_dir}")

# Inspect 'val' split parent directory
val_split_dir = RAW_ROOT / 'VisDrone2019-DET-val'
print(f"\nChecking contents of validation split directory: {val_split_dir}")
if val_split_dir.exists():
    print(f"  Contents: {os.listdir(val_split_dir)}")
else:
    print(f"  Directory does not exist: {val_split_dir}")

print(f"\n--- End of Inspection ---")

--- Inspecting Raw VisDrone Dataset Structure ---

Checking contents of training split directory: /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-train
  Contents: ['images', 'labels']

Checking contents of validation split directory: /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-val
  Contents: ['images', 'labels']

--- End of Inspection ---


In [ ]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00


In [ ]:
for split in ['train', 'val']:
    (CUSTOM_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (CUSTOM_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

In [ ]:
CLASS_MAP = {
    0: 0,  # Raw YOLO 0 (typically VisDrone 1: pedestrian) -> human
    1: 0,  # Raw YOLO 1 (typically VisDrone 2: people)     -> human

    3: 1,  # Raw YOLO 3 (typically VisDrone 4: car)        -> car
    4: 1,  # Raw YOLO 4 (typically VisDrone 5: van)        -> car
    5: 1,  # Raw YOLO 5 (typically VisDrone 6: truck)      -> car
    8: 1,  # Raw YOLO 8 (typically VisDrone 9: bus)        -> car

    2: 2,  # Raw YOLO 2 (typically VisDrone 3: bicycle)    -> other
    6: 2,  # Raw YOLO 6 (typically VisDrone 7: tricycle)   -> other
    7: 2,  # Raw YOLO 7 (typically VisDrone 8: awning-tricycle) -> other
    9: 2,  # Raw YOLO 9 (typically VisDrone 10: motor)     -> other
    10: 2, # Raw YOLO 10 (typically VisDrone 11: others)   -> other
    # Raw YOLO for VisDrone 12 (ignored regions) is often not included or mapped to -1 and thus skipped
}

CLASS_NAMES = {
    0: 'human',
    1: 'car',
    2: 'other' # New class for previously unmapped categories
}

In [ ]:
from PIL import Image
from pathlib import Path
import shutil

def convert_split(split):
    # Correctly specify the path for VisDrone dataset splits
    visdrone_split_folder = f'VisDrone2019-DET-{split}'

    img_dir = RAW_ROOT / visdrone_split_folder / 'images'
    ann_dir = RAW_ROOT / visdrone_split_folder / 'labels' # This directory now contains YOLO format labels

    out_img_dir = CUSTOM_ROOT / 'images' / split
    out_lbl_dir = CUSTOM_ROOT / 'labels' / split

    # Create output directories if they don't exist
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing images from: {img_dir}")
    print(f"Processing annotations (YOLO format) from: {ann_dir}")
    print(f"Output images to: {out_img_dir}")
    print(f"Output labels to: {out_lbl_dir}")

    processed_count = 0
    skipped_count = 0

    for img_path in img_dir.glob('*.jpg'):
        ann_path = ann_dir / f'{img_path.stem}.txt'

        if not ann_path.exists():
            print(f"Warning: Annotation file not found for {img_path.name}. Skipping image and its annotations.")
            skipped_count += 1
            continue

        yolo_lines_filtered = []

        try:
            with open(ann_path, 'r') as f:
                for line in f:
                    parts = line.strip().split(' ') # Split by space, as inspection showed YOLO format
                    if len(parts) != 5: # Expecting 5 parts: class, x_c, y_c, w, h
                        # Removed debug print, as the issue is understood to be class mapping
                        continue

                    try:
                        original_category = int(parts[0])
                    except ValueError:
                        # Removed debug print, as the issue is understood to be class mapping
                        continue

                    if original_category in CLASS_MAP: # Filter for desired classes
                        new_category = CLASS_MAP[original_category]
                        # Keep original normalized coordinates
                        xc, yc, wn, hn = parts[1:]
                        yolo_lines_filtered.append(f'{new_category} {xc} {yc} {wn} {hn}')
                    else:
                        # Removed debug print, as the issue is understood to be class mapping
                        pass # Object is correctly skipped if not in CLASS_MAP

        except Exception as e:
            print(f"Error processing annotation file {ann_path.name}: {e}. Skipping image and its annotations.")
            skipped_count += 1
            continue


        if yolo_lines_filtered: # Only copy if there are valid annotations after filtering
            (out_lbl_dir / f'{img_path.stem}.txt').write_text('\n'.join(yolo_lines_filtered))
            shutil.copy2(img_path, out_img_dir / img_path.name)
            processed_count += 1
        else:
            # Removed debug print, as the issue is understood to be class mapping
            skipped_count += 1

    print(f"Finished processing split '{split}'. Processed {processed_count} images, skipped {skipped_count} images.")

# Call the function for both train and val splits
convert_split('train')
convert_split('val')

Processing images from: /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/images
Processing annotations (YOLO format) from: /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/labels
Output images to: /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/train
Output labels to: /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/labels/train


KeyboardInterrupt: 

In [ ]:
import os

print("--- Inspecting Sample Annotation File and Class Map ---\n")

# Get one image that was skipped (from previous output)
# You might need to change this if the exact image name changes in your output
problem_image_name = '0000002_00005_d_0000014.jpg'
problem_image_stem = Path(problem_image_name).stem

# Construct the path to its original annotation file in the RAW_ROOT
train_split_folder = 'VisDrone2019-DET-train'
raw_ann_file_path = RAW_ROOT / train_split_folder / 'labels' / f'{problem_image_stem}.txt'

print(f"Attempting to read annotation file: {raw_ann_file_path}")

if raw_ann_file_path.exists():
    print("Annotation file content:")
    with open(raw_ann_file_path, 'r') as f:
        for i, line in enumerate(f):
            print(f"  Line {i+1}: {line.strip()}")
            if i >= 9: # Print first 10 lines to avoid too much output
                print("  ... (truncated)")
                break
else:
    print(f"Error: Annotation file not found at {raw_ann_file_path}")

print(f"\nCurrently defined CLASS_MAP: {CLASS_MAP}")
print(f"Currently defined CLASS_NAMES: {CLASS_NAMES}")

print("\n--- End of Inspections ---")


--- Inspecting Sample Annotation File and Class Map ---

Attempting to read annotation file: /content/drive/MyDrive/Antlings_Visdrone/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/labels/0000002_00005_d_0000014.txt
Annotation file content:
  Line 1: 3 0.776042 0.902778 0.077083 0.061111
  Line 2: 3 0.697396 0.829630 0.063542 0.085185
  Line 3: 3 0.652083 0.786111 0.066667 0.094444
  Line 4: 3 0.617188 0.757407 0.063542 0.070370
  Line 5: 3 0.596354 0.719444 0.067708 0.061111
  Line 6: 3 0.570833 0.649074 0.070833 0.064815
  Line 7: 3 0.555208 0.615741 0.066667 0.057407
  Line 8: 3 0.545312 0.579630 0.046875 0.051852
  Line 9: 3 0.534375 0.550926 0.050000 0.050000
  Line 10: 3 0.507292 0.512037 0.050000 0.053704
  ... (truncated)

Currently defined CLASS_MAP: {0: 0, 1: 0, 3: 1, 4: 1, 5: 1, 8: 1, 2: 2, 6: 2, 7: 2, 9: 2, 10: 2}
Currently defined CLASS_NAMES: {0: 'human', 1: 'car', 2: 'other'}

--- End of Inspections ---


In [ ]:
import yaml

data = {
    'path': str(CUSTOM_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'human',
        1: 'car',
        2: 'other' # Add the new 'other' class to data.yaml
    }
}

with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(YAML_PATH.read_text())

path: /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom
train: images/train
val: images/val
names:
  0: human
  1: car
  2: other



In [ ]:
import os

# Verify the contents of the custom dataset directories
print(f"--- Verifying Custom Dataset Directories ---")

train_images_path = CUSTOM_ROOT / 'images' / 'train'
train_labels_path = CUSTOM_ROOT / 'labels' / 'train'
val_images_path = CUSTOM_ROOT / 'images' / 'val'
val_labels_path = CUSTOM_ROOT / 'labels' / 'val'

def list_dir_contents(path, max_items=5):
    if path.exists():
        contents = os.listdir(path)
        print(f"  {path} ({len(contents)} items):")
        for i, item in enumerate(contents):
            if i < max_items:
                print(f"    - {item}")
            else:
                print(f"    ... ({len(contents) - max_items} more items)")
                break
    else:
        print(f"  Directory does not exist: {path}")

print("\nChecking training images directory:")
list_dir_contents(train_images_path)

print("\nChecking training labels directory:")
list_dir_contents(train_labels_path)

print("\nChecking validation images directory:")
list_dir_contents(val_images_path)

print("\nChecking validation labels directory:")
list_dir_contents(val_labels_path)

print(f"\n--- End of Verifications ---")

--- Verifying Custom Dataset Directories ---

Checking training images directory:
  /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/train (6471 items):
    - 9999940_00000_d_0000044.jpg
    - 9999940_00000_d_0000045.jpg
    - 9999940_00000_d_0000046.jpg
    - 9999940_00000_d_0000047.jpg
    - 9999940_00000_d_0000048.jpg
    ... (6466 more items)

Checking training labels directory:
  /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/labels/train (6471 items):
    - 9999940_00000_d_0000050.txt
    - 9999940_00000_d_0000051.txt
    - 9999940_00000_d_0000052.txt
    - 9999940_00000_d_0000053.txt
    - 9999940_00000_d_0000054.txt
    ... (6466 more items)

Checking validation images directory:
  /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val (548 items):
    - 0000103_04513_d_0000034.jpg
    - 0000024_01543_d_0000015.jpg
    - 0000103_04948_d_0000035.jpg
    - 0000242_00843_d_0000004.jpg
    - 0000026_01500_d_0000027.jpg
    ... (543 more items)


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=str(YAML_PATH),
    epochs=32,
    imgsz=640,
    batch=16,
    project=str(CUSTOM_ROOT / 'runs'),
    name='visdrone_human_car'
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Antlings_Visdrone/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_human_car-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

In [ ]:
from ultralytics import YOLO

best = YOLO(str(CUSTOM_ROOT / 'runs' / 'visdrone_human_car-4' / 'weights' / 'best.pt'))
best.val(data=str(YAML_PATH))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 0.7±0.3 MB/s, size: 170.7 KB)
val: Scanning /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 95.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.2s/it 1:53
                   all        548      38759      0.609      0.461      0.479      0.247
                 human        531      13969      0.568    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dd16a0c1280>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
results = best.predict(source=str(CUSTOM_ROOT / 'images' / 'val'), conf=0.25, save=True)

for r in results[:5]:
    cls = r.boxes.cls.cpu().numpy().astype(int)
    human_count = (cls == 0).sum()
    car_count = (cls == 1).sum()
    print('humans:', human_count, 'cars:', car_count)


image 1/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_02999_d_0000005.jpg: 384x640 16 humans, 11 cars, 22 others, 182.7ms
image 2/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_03499_d_0000006.jpg: 384x640 8 humans, 19 cars, 13 others, 166.4ms
image 3/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_03999_d_0000007.jpg: 384x640 16 humans, 4 cars, 20 others, 186.6ms
image 4/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_04527_d_0000008.jpg: 384x640 20 humans, 3 cars, 28 others, 182.8ms
image 5/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_05249_d_0000009.jpg: 384x640 17 humans, 98 cars, 7 others, 180.7ms
image 6/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/images/val/0000001_05499_d_0000010.jpg: 384x640 4 humans, 82 cars, 7 others, 369.5ms
image 7/548 /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/

In [ ]:
import matplotlib.pyplot as plt
metrics = best.val(data=str(YAML_PATH), plots=True)
conf_matrix_obj = metrics.confusion_matrix
fig = conf_matrix_obj.plot(
    save_dir=metrics.save_dir
)
plt.show()
print(f"\nThe confusion matrix plot has  been saved to: {metrics.save_dir}")

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
val: Fast image access ✅ (ping: 0.9±0.7 ms, read: 31.0±16.9 MB/s, size: 140.7 KB)
val: Scanning /content/drive/MyDrive/Antlings_Visdrone/visdrone_custom/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 88.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.8s/it 1:40
                   all        548      38759      0.609      0.461      0.479      0.247
                 human        531      13969      0.568      0.339       0.35      0.128
                   car        519      17040      0.702      0.729       0.75      0.478
                 other        510       7750      0.558      0.316      0.337      0.135
Speed: 2.0ms preprocess, 143.0ms inference, 0.0ms loss, 6.7ms postprocess per image
Results saved to /content/runs/detect/val-3

The confusion matrix plot has  been saved to: /conte